# TraceLens: train the AI image classifier (Kaggle, free GPU)

This notebook runs the whole model pipeline from the PRD:

1. Build a manifest from the datasets
2. FFT + logistic regression baseline (week 2)
3. Fine-tune EfficientNet-B0, and ResNet50 for comparison (week 3)
4. Temperature calibration
5. ONNX export with the Grad-CAM output, checked against PyTorch
6. Evaluation through the real serving code: test set, unseen generators, your holdout set, robustness
7. Publish to the Hugging Face Model Hub

**Before you run it**

* Settings (right panel) → **Accelerator: GPU T4 x2** (or P100) and **Internet: On**.
* **Add Input** → add the datasets you will use, for example:
  * CIFAKE: `birdy654/cifake-real-and-ai-generated-synthetic-images`
  * a GenImage subset (search "GenImage" in Kaggle datasets; pick one with one folder per generator)
  * your own holdout set, uploaded as a private Kaggle dataset with `real/` and `ai/<generator>/` folders
* Add-ons → **Secrets** → add `HF_TOKEN` (a Hugging Face *write* token) if you want to publish the model.

In [ ]:
# 1. Get the code (replace with your GitHub repo URL)
REPO_URL = "https://github.com/YOUR-USERNAME/TraceLens.git"
!git clone -q $REPO_URL /kaggle/working/TraceLens
%cd /kaggle/working/TraceLens/training
!pip install -q -r requirements.txt

In [ ]:
# 2. Point these at the datasets you added (leave a path as None to skip that dataset)
import os
CIFAKE   = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"
GENIMAGE = None   # e.g. "/kaggle/input/<genimage-subset>"  (one folder per generator)
HOLDOUT  = None   # e.g. "/kaggle/input/<your-holdout-set>"
CASIA    = None   # e.g. "/kaggle/input/<casia-v2>"  (Au/ and Tp/ folders)

# Generators to hold out of training, to measure accuracy on generators the model never saw.
# Use folder names from your GenImage copy, e.g. "Midjourney,wukong". Empty = no cross-generator test.
UNSEEN = ""

os.environ["TRACELENS_ARTIFACTS"] = "/kaggle/working/artifacts"
flags = " ".join(f"--{k} {v}" for k, v in [("cifake", CIFAKE), ("genimage", GENIMAGE), ("holdout", HOLDOUT), ("casia", CASIA)] if v)
!python -m tracelens_train.prepare_data $flags --limit-per-class 15000

## Week 2: frequency-domain baseline

In [ ]:
!python -m tracelens_train.fft_baseline --max-train 20000

## Week 3: fine-tune EfficientNet-B0 (main model) and ResNet50 (comparison)

In [ ]:
# About 20-40 min on a T4 for 30k images x 8 epochs. Lower --max-train for a quick first run.
!python -m tracelens_train.train --arch efficientnet_b0 --epochs 8 --batch-size 64 --exclude-generators "$UNSEEN"

In [ ]:
!python -m tracelens_train.train --arch resnet50 --epochs 8 --batch-size 48 --exclude-generators "$UNSEEN"

## Calibration and ONNX export

In [ ]:
!python -m tracelens_train.calibrate --run efficientnet_b0
!python -m tracelens_train.export_onnx --run efficientnet_b0 --name efficientnet_b0_v1

## Evaluation (through the exact serving code path)

In [ ]:
!python -m tracelens_train.evaluate --unseen "$UNSEEN"

In [ ]:
# Optional: how well the ELA + noise + metadata manipulation score separates CASIA v2 authentic vs tampered
if CASIA:
    !python -m tracelens_train.evaluate_forensics

## Compare runs in MLflow

MLflow logs are in `/kaggle/working/TraceLens/training/mlflow.db` (plus `mlruns/`). Download
`mlflow_logs.zip` from the Output tab, unzip it into `training/` on your computer and run
`mlflow ui --backend-store-uri sqlite:///mlflow.db` to compare
EfficientNet-B0, ResNet50 and the FFT baseline side by side.

In [ ]:
!cd /kaggle/working/TraceLens/training && zip -qr /kaggle/working/mlflow_logs.zip mlflow.db mlruns
!cp -r /kaggle/working/artifacts/export /kaggle/working/model_export
!ls -la /kaggle/working/model_export

## Publish to the Hugging Face Model Hub

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
HF_MODEL_REPO = "YOUR-USERNAME/tracelens-detector"
!python -m tracelens_train.push_to_hub --repo $HF_MODEL_REPO

Then set `HF_MODEL_REPO` in your Hugging Face Space settings (Variables) to the repo above and
restart the Space. `/api/v1/health` will show the model name and the published metrics.